<a href="https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: A page should be prioritised for review if it has meaningful search visibility but appears to have an opportunity for improvement. I will give higher scores to pages with substantial impressions and relatively weak click-through performance, while also considering whether the page has a poor search position. The rule is deliberately simple and transparent so that it provides a fair baseline for comparison with the ML model.

Reason codes:

high_visibility_low_ctr — the page receives a substantial number of impressions but has a relatively low CTR.

poor_search_position — the page has a relatively weak average search position.

high_visibility_poor_position — the page has substantial impressions while also having a relatively poor search position.

low_visibility — the page has too few impressions to provide strong evidence that it should be prioritised.

no_gsc_data — GSC data is unavailable, so search performance cannot be reliably assessed.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""")

march_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,

        BOOL_OR(gsc_data_available) AS gsc_data_available,
        BOOL_OR(ga4_data_available) AS ga4_data_available

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

import numpy as np

march_df["gsc_ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
).fillna(0)

march_df["engagement_rate"] = (
    march_df["ga4_engaged_sessions"] /
    march_df["ga4_sessions"].replace(0, np.nan)
).fillna(0)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_ctr",
    "ga4_sessions",
    "engagement_rate"
]

X = (
    march_df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
# Start with a copy so the original feature data is unchanged
baseline_df = march_df.copy()

# Reason-code conditions
baseline_df["high_visibility"] = (
    baseline_df["gsc_impressions"] >= 500
)

baseline_df["low_ctr"] = (
    baseline_df["gsc_ctr"] < 0.01
)

baseline_df["poor_position"] = (
    baseline_df["gsc_avg_position"] > 20
)

baseline_df["no_gsc_data"] = (
    baseline_df["gsc_data_available"] == False
)

# Transparent baseline score
# Higher score = higher priority for review
baseline_df["baseline_score"] = (
    baseline_df["high_visibility"].astype(int) *
    baseline_df["low_ctr"].astype(int) *
    baseline_df["gsc_impressions"]
)

# Reason codes
def get_reason(row):
    if row["no_gsc_data"]:
        return "no_gsc_data"
    elif row["high_visibility"] and row["low_ctr"] and row["poor_position"]:
        return "high_visibility_poor_position"
    elif row["high_visibility"] and row["low_ctr"]:
        return "high_visibility_low_ctr"
    elif row["poor_position"]:
        return "poor_search_position"
    else:
        return "low_priority"

baseline_df["reason_code"] = baseline_df.apply(get_reason, axis=1)

# Rank highest-scoring pages first
baseline_df = baseline_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_df["rank"] = baseline_df.index + 1

# Keep the useful fields in the output
output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_ctr",
    "gsc_avg_position"
]

baseline_output = baseline_df[output_cols]

# Save the ranked queue
import os

os.makedirs("work/outputs", exist_ok=True)

baseline_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(f"Saved {len(baseline_output):,} rows")
print(baseline_output.head(20))

Saved 331,437 rows
    rank           client_hash_id           content_hash_id  baseline_score  \
0      1  client_e547b89c05043229  content_eadb33b5df496f4a        617124.0   
1      2  client_e547b89c05043229  content_ec2e0346994fb5a5        245276.0   
2      3  client_23a62021009f63c4  content_e8a52cf3d5988c07        244931.0   
3      4  client_e547b89c05043229  content_0e03de7680314cd5        221310.0   
4      5  client_23a62021009f63c4  content_44f34c0a90047651        212404.0   
5      6  client_62f4a7e64f5e0096  content_7172a7fad43f0998        205867.0   
6      7  client_e547b89c05043229  content_8d7d99f109e19aa2        203497.0   
7      8  client_62f4a7e64f5e0096  content_f107e54b10b43725        195997.0   
8      9  client_23a62021009f63c4  content_36e53e9c707674fc        194579.0   
9     10  client_62f4a7e64f5e0096  content_b99ea6861864dea5        194337.0   
10    11  client_e547b89c05043229  content_4ffe18112a5642e3        186983.0   
11    12  client_62f4a7e64f5e0096

In [6]:
import os

size_mb = os.path.getsize(
    "work/outputs/baseline_action_score.csv"
) / (1024**2)

print(f"File size: {size_mb:.2f} MB")

File size: 31.96 MB


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Rank | Action                          | Reason code                     | Confidence note                                                                  | What would make it wrong                                                                                 |
| ---: | ------------------------------- | ------------------------------- | -------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------- |
|    1 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (617,124 impressions) with CTR below 1%                     | High impressions may reflect branded searches or a query mix where low CTR is expected                   |
|    2 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (245,276 impressions) and 0.60% CTR                         | Low CTR may be normal for the queries generating these impressions                                       |
|    3 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (244,931 impressions) and 0.27% CTR                         | The page may already be performing appropriately for its search intent                                   |
|    4 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (221,310 impressions) and 0.33% CTR                         | Low CTR may be caused by SERP features or query intent rather than poor content                          |
|    5 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (212,404 impressions) and extremely low CTR (0.011%)        | The impressions could come from searches where users rarely click organic results                        |
|    6 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (205,867 impressions) and 0.42% CTR                         | Search intent may not match the page, so refreshing content may not solve the problem                    |
|    7 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (203,497 impressions) and 0.14% CTR                         | Low CTR may be explained by the type of queries rather than page quality                                 |
|    8 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (195,997 impressions) and 0.51% CTR                         | The page may be visible for many low-intent or irrelevant queries                                        |
|    9 | Review page for content refresh | `high_visibility_poor_position` | High visibility (194,579 impressions) combined with poor average position (32.8) | The page may rank poorly because the queries are highly competitive, not because its content is outdated |
|   10 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (194,337 impressions) and 0.19% CTR                         | SERP features or search intent may explain the low CTR                                                   |
|   11 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (186,983 impressions) and 0.31% CTR                         | Low CTR may be appropriate for the page's query mix                                                      |
|   12 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (170,808 impressions) and 0.15% CTR                         | The page may not be a suitable candidate for improvement despite the low CTR                             |
|   13 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (164,885 impressions) and 0.24% CTR                         | Low CTR could be caused by search intent or SERP layout                                                  |
|   14 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (152,806 impressions) and 0.62% CTR                         | The CTR may be reasonable for the queries involved                                                       |
|   15 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (151,166 impressions) and 0.27% CTR                         | There may be no actionable content problem behind the low CTR                                            |
|   16 | Review page for content refresh | `high_visibility_poor_position` | High visibility (143,907 impressions) and poor average position (22.6)           | Competition or query intent may explain the position rather than content quality                         |
|   17 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (143,019 impressions) and extremely low CTR (0.03%)         | The page could be appearing for queries where organic clicks are uncommon                                |
|   18 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (142,304 impressions) and 0.24% CTR                         | Low CTR may reflect the query mix rather than an outdated page                                           |
|   19 | Review page for content refresh | `high_visibility_poor_position` | High visibility (140,156 impressions) and poor average position (23.3)           | The page may require SEO changes rather than a content refresh                                           |
|   20 | Review page for content refresh | `high_visibility_low_ctr`       | Very high visibility (139,417 impressions) and 0.14% CTR                         | Low CTR alone does not prove that refreshing the content will improve performance                        |


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Some of the top-ranked pages may be weak recommendations because the baseline treats high impressions and low CTR as evidence that a page should be reviewed. A low CTR does not necessarily mean that the content is poor or outdated; it could be caused by search intent, SERP features, branded searches, or the type of queries the page appears for. Pages with high impressions but good search positions may therefore receive a high score even when there is no obvious content problem. Similarly, a poor search position may require an SEO or technical change rather than a content refresh.

Leakage check: The baseline uses only information available during the March 2026 observation window: GSC impressions, clicks, CTR, average position, and the GSC availability flag. I did not use future-month outcomes, future traffic, future trend values, or any label-derived columns. I also did not use existing product scores or prioritisation flags as inputs. Therefore, the baseline does not deliberately include product-flag or future-window leakage. The later ML model will require a stricter leakage check because engineered features and the eventual outcome must be separated in time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.